In [1]:
# -*- coding: utf-8 -*-
"""端到端量价时序预测 -- v1 推理侧脚本 (加载本地模型 + 推理打分)。

本文件只负责**推理**: 加载训练侧脚本 `transformer_train.py` 产出的 `transformer_model.json`,
在平台注入的**测试集区间**上打分, **不训练**。

训练逻辑 (配置 / 模型结构 / 数据构建 / `train_and_save`) 全部沉淀在 `transformer_train.py` 里,
作为单一事实来源。流程拆成两个阶段:
1. 阶段一 (参赛者本地/平台运行一次): 调 `train_and_save(...)`, 在写死训练区间从零训练,
   产出 `transformer_model.json` (权重 + 标准化统计 + 结构超参, 纯文本 JSON)。
2. 阶段二 (平台公榜调用 `main`): 直接加载 `transformer_model.json`, 在平台注入的测试区间上推理打分。

提交时请把 `transformer_train.py` 与训练好的 `transformer_model.json` 随 notebook 一并上传。
公榜阶段平台只替换 `datasources / start_date / end_date` 并调用 `main`, 仅基于提交权重推理;
私榜阶段平台用 `train_and_save` 在隔离环境从零重训。

推理复用训练侧的 `build_windows`, 且标准化统计 (mean/std) 随权重一起存盘、推理时直接复用,
保证两阶段预处理严格一致, 杜绝数据泄漏与 train/infer 漂移。
回看窗口在测试区间起始处无历史时左侧 padding + mask, 保证评估区间每个交易日都有分数。
"""
import os

import numpy as np
import pandas as pd
import dai
import torch
import structlog

from transformer_train import (
    MODEL_PATH, BATCH,
    StockTransformer, build_windows, load_bars, pool, train_and_save, load_model,
    set_features,
)

logger = structlog.get_logger()


def main(datasources, start_date, end_date):
    """加载已训练好的模型, 在样本外测试区间 (start_date~end_date) 上推理打分。

    本函数**不训练**: 权重来自随 notebook 上传的 MODEL_PATH 文件。
    start_date~end_date 为平台注入的【测试集区间】, 输出每日分数 ['date','instrument','score']。"""
    table = datasources["bar5m"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"未找到模型文件 {MODEL_PATH}; 请先运行 train_and_save(...) 训练并保存, 再随 notebook 一起上传")

    ckpt = load_model(MODEL_PATH, map_location=device)
    set_features(ckpt["feature_cols"], ckpt["price_cols"], ckpt["vol_cols"])  # 用存盘特征, 跳过注入表探测
    stats = (np.asarray(ckpt["mean"], np.float32), np.asarray(ckpt["std"], np.float32))
    model = StockTransformer(**ckpt["model_cfg"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    logger.info("已加载模型", path=MODEL_PATH, device=str(device))

    # ---------- 推理 (样本外测试区间, 用平台注入表) ----------
    logger.info("构建测试集并预测", table=table, start=str(start_date), end=str(end_date))
    instruments = pool(start_date, end_date)
    raw = load_bars(table, start_date, end_date, instruments)  # 注入表只含测试区间, 前推 buffer 为空
    Xte, Mte, idx_df, _ = build_windows(raw, start_date, end_date, "infer", stats)

    preds = []
    Xte_t = torch.from_numpy(Xte)
    Mte_t = torch.from_numpy(Mte)
    with torch.no_grad():
        for i in range(0, len(idx_df), BATCH):
            xb = Xte_t[i:i + BATCH].to(device)
            mb = Mte_t[i:i + BATCH].to(device)
            preds.append(model(xb, mb).cpu().numpy())
    idx_df["score"] = np.concatenate(preds).astype(np.float64)

    # ---------- 对齐中证 1000 + 规范输出 ----------
    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={"date": [start_date, end_date]}).df()
    result = (pd.merge(idx_df, stk, on=["date", "instrument"], how="inner")
                .replace([np.inf, -np.inf], np.nan).dropna(subset=["score"])
                .drop_duplicates(["date", "instrument"])[["date", "instrument", "score"]]
                .reset_index(drop=True))
    logger.info("分数构建完成", rows=len(result), days=result["date"].nunique(),
                instruments=result["instrument"].nunique())
    return result


if __name__ == "__main__":
    from bigmodule import M

    datasources = {"bar5m": "bigalpha_2026_stock_bar5m"}

    # 本地首次运行: 若权重文件不存在, 先训练并保存
    if not os.path.exists(MODEL_PATH):
        logger.info("未发现已保存模型, 开始训练", path=MODEL_PATH)
        train_and_save(datasources)

    # 本地用 2024 全年模拟「平台注入的测试集区间」
    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    logger.info("计算分数 (仅加载权重推理, 不重训)", start=start_date, end=end_date)
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())

    # 评估系统: 分数经风格剔除后等价于每日单因子, show=True 画绩效图 (IC / 分组 / 压力期)
    logger.info("开始评估分数")
    result = M.bigalpha_eval._latest(factor_data=score_data, show=True)


[2026-08-05 16:34:33] [info     ] 未发现已保存模型, 开始训练                 path=/home/aiuser/work/比赛/v2/transformer_model.json
[2026-08-05 16:34:33] [info     ] 训练设备                           device=cuda table=bigalpha_2026_stock_bar5m
[2026-08-05 16:34:33] [info     ] 成分股数                           n=1932
[2026-08-05 16:34:40] [info     ] 解析特征字段                         n_feat=27 price=['open', 'high', 'low', 'close', 'bid_price1', 'ask_price1', 'bid_price2', 'ask_price2', 'bid_price3', 'ask_price3', 'bid_price4', 'ask_price4', 'bid_price5', 'ask_price5', 'pre_close'] table=bigalpha_2026_stock_bar5m vol=['volume', 'amount', 'bid_volume1', 'ask_volume1', 'bid_volume2', 'ask_volume2', 'bid_volume3', 'ask_volume3', 'bid_volume4', 'ask_volume4', 'bid_volume5', 'ask_volume5']
